# Stage 7 — Separate no-weather model training

P95 is the primary experiment and P90 is a sensitivity experiment. Both use the same rows, dates, chronological split logic, model families, and tuning protocol. Hyperparameters are selected on a late holdout inside the training period. The official validation set is reserved for Stage 8 calibration and threshold selection; the test set remains untouched.

In [ ]:
from pathlib import Path
import json, platform, hashlib, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from IPython.display import display
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss, log_loss
from xgboost import XGBClassifier

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid")
ROOT=Path.cwd().resolve()
if not (ROOT/"data").exists(): ROOT=ROOT.parent
DATA=ROOT/"data"/"processed"
MODEL_DIR=ROOT/"models"; MODEL_DIR.mkdir(parents=True,exist_ok=True)
TABLE_DIR=ROOT/"reports"/"tables"/"07_model_training"; TABLE_DIR.mkdir(parents=True,exist_ok=True)
FIGURE_DIR=ROOT/"reports"/"figures"/"07_model_training"; FIGURE_DIR.mkdir(parents=True,exist_ok=True)
RANDOM_STATE=42
INNER_TRAIN_FRACTION_DATES=0.80
COMMON_FEATURES=["lag1_peak_kw","lag1_mean_kw","lag1_energy_kwh","lag1_coverage_ratio","lag1_was_complete","peak_mean_7d","peak_mean_30d","peak_max_7d","peak_max_30d","peak_std_7d","peak_std_30d","energy_mean_7d","energy_mean_30d","lag1_peak_over_p95","lag1_peak_minus_p95","peak_mean_7d_over_30d","lag1_peak_zscore_30d","day_of_week_sin","day_of_week_cos","is_weekend","month_sin","month_cos"]
EXPERIMENTS={
 "p95":{"path":DATA/"05_model_ready_m0_p95.parquet","role":"primary","features":COMMON_FEATURES+["p95_high_count_7d","p95_high_count_30d","p95_days_since_last_high"]},
 "p90":{"path":DATA/"05_model_ready_m0_p90.parquet","role":"sensitivity","features":COMMON_FEATURES+["p90_high_count_7d","p90_high_count_30d","p90_days_since_last_high"]},
}

def save_table(df,name,title=None):
    df.to_csv(TABLE_DIR/name,index=False)
    if title: print(title)
    display(df)

def class_weights(y):
    counts=pd.Series(y).value_counts(); n=len(y)
    return {0:n/(2*counts[0]),1:n/(2*counts[1])}

def metrics(y,p):
    eps=1e-7; p=np.clip(np.asarray(p,float),eps,1-eps)
    return {"pr_auc":average_precision_score(y,p),"roc_auc":roc_auc_score(y,p),"brier":brier_score_loss(y,p),"log_loss":log_loss(y,p,labels=[0,1])}

def make_pipeline(family,params,y_fit):
    imputer=SimpleImputer(strategy="median",add_indicator=True,keep_empty_features=True)
    weights=class_weights(y_fit)
    if family=="logistic_regression":
        model=LogisticRegression(C=params["C"],class_weight=weights,solver="liblinear",max_iter=2500,random_state=RANDOM_STATE)
        return Pipeline([("imputer",imputer),("scaler",StandardScaler()),("model",model)])
    if family=="random_forest":
        model=RandomForestClassifier(n_estimators=params["n_estimators"],max_depth=params["max_depth"],min_samples_leaf=5,max_features="sqrt",class_weight=weights,n_jobs=-1,random_state=RANDOM_STATE)
        return Pipeline([("imputer",imputer),("model",model)])
    if family=="xgboost":
        model=XGBClassifier(n_estimators=params["n_estimators"],max_depth=params["max_depth"],learning_rate=.05,subsample=.8,colsample_bytree=.8,min_child_weight=3,reg_lambda=1,scale_pos_weight=weights[1]/weights[0],eval_metric="logloss",tree_method="hist",n_jobs=4,random_state=RANDOM_STATE,verbosity=0)
        return Pipeline([("imputer",imputer),("model",model)])
    raise ValueError(family)

CANDIDATES={
 "logistic_regression":[{"C":.1},{"C":1.0}],
 "random_forest":[{"n_estimators":200,"max_depth":8},{"n_estimators":200,"max_depth":None}],
 "xgboost":[{"n_estimators":250,"max_depth":3},{"n_estimators":250,"max_depth":5}],
}
print("Official validation and test rows are excluded from this notebook's fitting and selection.")

## 7.1 Validate experiment symmetry and isolate training rows

In [ ]:
datasets={k:pd.read_parquet(v["path"]) for k,v in EXPERIMENTS.items()}
for df in datasets.values(): df["measurement_date"]=pd.to_datetime(df["measurement_date"])
validation=pd.DataFrame([
 ("same_rows",datasets["p95"]["row_id"].equals(datasets["p90"]["row_id"]),len(datasets["p95"])),
 ("same_splits",datasets["p95"]["split_name"].equals(datasets["p90"]["split_name"]),"ordered equality"),
 ("same_common_features",all(c in datasets["p95"] and c in datasets["p90"] for c in COMMON_FEATURES),len(COMMON_FEATURES)),
 ("no_weather",not any("weather" in c.lower() for df in datasets.values() for c in df.columns),"M0"),
 ("p95_target_binary",set(datasets["p95"].target.unique())=={0,1},sorted(datasets["p95"].target.unique())),
 ("p90_target_binary",set(datasets["p90"].target.unique())=={0,1},sorted(datasets["p90"].target.unique())),
],columns=["check","passed","evidence"])
save_table(validation,"07_01_input_validation.csv","Training input validation")
assert validation.passed.all(),validation.loc[~validation.passed]

split_counts=[]
for key,df in datasets.items():
    for split,g in df.groupby("split_name",sort=False): split_counts.append({"target":key.upper(),"split_name":split,"rows":len(g),"events":int(g.target.sum()),"event_rate":g.target.mean(),"start_date":g.measurement_date.min(),"end_date":g.measurement_date.max()})
save_table(pd.DataFrame(split_counts),"07_02_reserved_split_counts.csv","Reserved split counts")

## 7.2 Internal chronological tuning on training dates only

For each target, the earliest 80% of training dates form the inner fit set and the latest 20% form the inner tuning set. Every candidate pipeline learns median values, missingness indicators, scaling where applicable, and class weights from the inner fit rows only.

In [ ]:
candidate_rows=[]; family_winners={}; fitted_models={}; experiment_manifests={}
for target_key,spec in EXPERIMENTS.items():
    df=datasets[target_key]
    official_train=df.loc[df.split_name.eq("train")].sort_values(["measurement_date","household_id"]).copy()
    dates=np.array(sorted(official_train.measurement_date.unique()))
    cut=int(np.floor(len(dates)*INNER_TRAIN_FRACTION_DATES))
    inner_fit=official_train.loc[official_train.measurement_date.isin(dates[:cut])]
    inner_tune=official_train.loc[official_train.measurement_date.isin(dates[cut:])]
    assert inner_fit.measurement_date.max()<inner_tune.measurement_date.min()
    assert inner_fit.target.nunique()==2 and inner_tune.target.nunique()==2
    X_fit=inner_fit[spec["features"]]; y_fit=inner_fit.target.astype(int)
    X_tune=inner_tune[spec["features"]]; y_tune=inner_tune.target.astype(int)
    prevalence=float(y_fit.mean())
    base_metrics=metrics(y_tune,np.full(len(y_tune),prevalence))
    candidate_rows.append({"target":target_key.upper(),"family":"historical_prevalence","params":json.dumps({"prevalence":prevalence}),"inner_fit_rows":len(inner_fit),"inner_tune_rows":len(inner_tune),**base_metrics})
    for family,grid in CANDIDATES.items():
        for params in grid:
            pipe=make_pipeline(family,params,y_fit)
            pipe.fit(X_fit,y_fit)
            probability=pipe.predict_proba(X_tune)[:,1]
            candidate_rows.append({"target":target_key.upper(),"family":family,"params":json.dumps(params,sort_keys=True),"inner_fit_rows":len(inner_fit),"inner_tune_rows":len(inner_tune),**metrics(y_tune,probability)})
    current=pd.DataFrame(candidate_rows)
    current=current.loc[(current.target.eq(target_key.upper())) & ~current.family.eq("historical_prevalence")]
    winners=current.sort_values(["family","pr_auc"],ascending=[True,False]).groupby("family",as_index=False).first()
    family_winners[target_key]=winners
    X_full=official_train[spec["features"]]; y_full=official_train.target.astype(int)
    fitted_models[target_key]={}
    for _,winner in winners.iterrows():
        family=winner["family"]; params=json.loads(winner["params"])
        pipe=make_pipeline(family,params,y_full); pipe.fit(X_full,y_full)
        fitted_models[target_key][family]=pipe
        joblib.dump(pipe,MODEL_DIR/f"07_{target_key}_{family}_m0.joblib")
    selected=winners.sort_values("pr_auc",ascending=False).iloc[0]
    selected_family=selected["family"]
    tree_winner=winners.loc[winners.family.isin(["random_forest","xgboost"])].sort_values("pr_auc",ascending=False).iloc[0]
    joblib.dump(fitted_models[target_key][selected_family],MODEL_DIR/f"07_{target_key}_selected_model_m0.joblib")
    joblib.dump(fitted_models[target_key][tree_winner.family],MODEL_DIR/f"07_{target_key}_best_tree_model_m0.joblib")
    full_prevalence=float(y_full.mean())
    (MODEL_DIR/f"07_{target_key}_baseline.json").write_text(json.dumps({"training_prevalence":full_prevalence},indent=2),encoding="utf-8")
    manifest={"target":target_key.upper(),"experiment_role":spec["role"],"feature_set":"M0","weather_included":False,"features":spec["features"],"official_train_rows":len(official_train),"official_train_dates":{"start":str(official_train.measurement_date.min().date()),"end":str(official_train.measurement_date.max().date())},"inner_fit":{"rows":len(inner_fit),"start":str(inner_fit.measurement_date.min().date()),"end":str(inner_fit.measurement_date.max().date())},"inner_tune":{"rows":len(inner_tune),"start":str(inner_tune.measurement_date.min().date()),"end":str(inner_tune.measurement_date.max().date())},"selected_family":selected_family,"selected_internal_pr_auc":float(selected.pr_auc),"best_tree_family":str(tree_winner.family),"best_tree_internal_pr_auc":float(tree_winner.pr_auc),"class_weights_full_train":class_weights(y_full),"official_validation_used":False,"test_used":False,"random_state":RANDOM_STATE}
    (MODEL_DIR/f"07_{target_key}_training_manifest.json").write_text(json.dumps(manifest,indent=2,default=float),encoding="utf-8")
    experiment_manifests[target_key]=manifest

candidates=pd.DataFrame(candidate_rows)
save_table(candidates,"07_03_internal_tuning_results.csv","Internal chronological tuning results")

## 7.3 Model-family selection and saved training-only pipelines

In [ ]:
winner_table=[]
for key,winners in family_winners.items():
    winner_table.append(winners.copy())
winner_table=pd.concat(winner_table,ignore_index=True)
save_table(winner_table,"07_04_family_winners.csv","Best candidate within each model family")
selection=pd.DataFrame([{"target":k.upper(),"experiment_role":EXPERIMENTS[k]["role"],"selected_family":m["selected_family"],"internal_pr_auc":m["selected_internal_pr_auc"],"best_tree_family":m["best_tree_family"],"best_tree_internal_pr_auc":m["best_tree_internal_pr_auc"]} for k,m in experiment_manifests.items()])
save_table(selection,"07_05_selected_models.csv","Selected models (P95 remains primary regardless of score)")

fig,ax=plt.subplots(figsize=(9,5))
sns.barplot(data=winner_table,x="family",y="pr_auc",hue="target",ax=ax,palette={"P95":"#b2182b","P90":"#2166ac"})
ax.set(title="Training-only chronological tuning PR-AUC",xlabel="",ylabel="PR-AUC")
plt.xticks(rotation=15); plt.tight_layout(); plt.savefig(FIGURE_DIR/"07_01_internal_tuning_pr_auc.png",dpi=180,bbox_inches="tight"); plt.show()

## 7.4 Final training-stage controls

In [ ]:
checks=[]
for key,spec in EXPERIMENTS.items():
    manifest=experiment_manifests[key]
    checks.extend([
      (key,"validation_not_used",manifest["official_validation_used"] is False),
      (key,"test_not_used",manifest["test_used"] is False),
      (key,"three_fitted_families",len(fitted_models[key])==3),
      (key,"selected_model_saved",(MODEL_DIR/f"07_{key}_selected_model_m0.joblib").exists()),
      (key,"tree_model_saved",(MODEL_DIR/f"07_{key}_best_tree_model_m0.joblib").exists()),
      (key,"pipeline_imputation",all("imputer" in p.named_steps for p in fitted_models[key].values())),
      (key,"target_specific_history",not any(c.startswith("p90_") for c in spec["features"]) if key=="p95" else not any(c.startswith("p95_") for c in spec["features"])),
    ])
final_checks=pd.DataFrame(checks,columns=["target","check","passed"])
save_table(final_checks,"07_06_final_validation.csv","Stage 7 final validation")
assert final_checks.passed.all(),final_checks.loc[~final_checks.passed]
run_manifest={"stage":"07_model_training","primary_target":"P95","sensitivity_target":"P90","selection_metric":"internal chronological PR-AUC","official_validation_used":False,"test_used":False,"models":selection.to_dict("records"),"software":{"python":platform.python_version(),"pandas":pd.__version__,"sklearn":__import__('sklearn').__version__,"xgboost":__import__('xgboost').__version__}}
(TABLE_DIR/"07_run_manifest.json").write_text(json.dumps(run_manifest,indent=2,default=str),encoding="utf-8")
print("Stage 7 complete. Stage 8 may now use official validation for calibration and threshold selection.")

## Interpretation boundary

The P90 experiment is retained only as sensitivity evidence. It is not promoted over P95 if it obtains a higher score. All saved pipelines are still uncalibrated and have not seen official validation or test data.